In [ ]:
!pip install biopython

In [ ]:
!pip install biopython torchac -q

In [ ]:
!apt-get update -qq
!apt-get install ninja-build -y
!pip install ninja -q
!pip install torchac biopython -q

In [ ]:
# =========================================================
# IMPROVED HYBRID TRANSFORMER DNA COMPRESSOR
# FULLY FIXED TORCHAC VERSION
# FLASH ATTENTION + KV CACHE + LOSSLESS
# =========================================================

# pip install -q torchac

import math
import time
import gzip
import bz2
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import torchac

# =========================================================
# REPRODUCIBILITY
# =========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# =========================================================
# CUDA OPTIMIZATION
# =========================================================

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.set_float32_matmul_precision("high")

# =========================================================
# CONFIG
# =========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", DEVICE)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

SEQ_LEN = 2048
STRIDE = 256
BATCH_SIZE = 32

EPOCHS = 40
LR = 3e-4
PATIENCE = 8

MODEL_PATH = "best_model.pt"

MAX_CACHE = 4096
CHUNK_SIZE = 256

# =========================================================
# DNA VOCAB
# =========================================================

DNA_MAP = {
    'A':0,
    'C':1,
    'G':2,
    'T':3
}

REV_DNA_MAP = {
    0:'A',
    1:'C',
    2:'G',
    3:'T'
}

VOCAB_SIZE = 4

# =========================================================
# CLEAN DNA
# =========================================================

def clean_seq(seq):

    return ''.join(
        c for c in seq.upper()
        if c in "ACGT"
    )

# =========================================================
# FILE LOADING
# =========================================================

def load_multiple_files(files):

    full_seq = ""

    for path in files:

        print("Loading:", path)

        with open(path) as f:

            seq = ''.join(
                line.strip()
                for line in f
            )

        seq = clean_seq(seq)

        full_seq += seq

    return full_seq

def load_fasta(path):

    with open(path) as f:

        seq = ''.join(
            line.strip()
            for line in f
        )

    seq = clean_seq(seq)

    return seq


# =========================================================
# DATASET
# =========================================================
class SequenceDataset(Dataset):

    def __init__(
        self,
        seq,
        seq_len,
        stride,
        species_id=0
    ):

        self.seq = seq
        self.seq_len = seq_len
        self.stride = stride
        self.species_id = species_id

    def __len__(self):

        if len(self.seq) <= self.seq_len:
            return 0

        return max(
            0,
            (
                len(self.seq)
                - self.seq_len
                - 1
            ) // self.stride
        )

    def __getitem__(self, idx):

        idx *= self.stride

        chunk = self.seq[
            idx :
            idx + self.seq_len + 1
        ]

        x = torch.tensor(
            [DNA_MAP[c] for c in chunk[:-1]],
            dtype=torch.long
        )

        y = torch.tensor(
            [DNA_MAP[c] for c in chunk[1:]],
            dtype=torch.long
        )

        species_id = torch.tensor(
            self.species_id,
            dtype=torch.long
        )

        return x, y, species_id
# =========================================================
# ROTARY POSITION EMBEDDING (RoPE)
# =========================================================

class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        dim,
        base=10000
    ):

        super().__init__()

        self.dim = dim

        inv_freq = 1.0 / (

            base ** (

                torch.arange(
                    0,
                    dim,
                    2
                ).float()

                / dim

            )

        )

        self.register_buffer(
            "inv_freq",
            inv_freq
        )

    def forward(
        self,
        seq_len,
        device
    ):

        positions = torch.arange(
            seq_len,
            device=device
        ).float()

        freqs = torch.einsum(
            "i,j->ij",
            positions,
            self.inv_freq
        )

        emb = torch.cat(
            [
                freqs,
                freqs
            ],
            dim=-1
        )

        return emb


def rotate_half(x):

    x_even = x[..., ::2]

    x_odd = x[..., 1::2]

    rotated = torch.stack(

        (

            -x_odd,

            x_even

        ),

        dim=-1

    )

    return rotated.flatten(-2)


def apply_rope(
    q,
    k,
    freqs
):

    cos = freqs.cos()

    sin = freqs.sin()

    cos = cos.unsqueeze(0).unsqueeze(0)

    sin = sin.unsqueeze(0).unsqueeze(0)

    q = (

        q * cos

        +

        rotate_half(q) * sin

    )

    k = (

        k * cos

        +

        rotate_half(k) * sin

    )

    return q, k
# =========================================================
# FLASH ATTENTION BLOCK + RoPE
# =========================================================

from torch.nn.functional import scaled_dot_product_attention

class FlashAttentionBlock(nn.Module):

    def __init__(
        self,
        dim,
        heads,
        ff_dim
    ):

        super().__init__()

        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads

        self.rope = RotaryEmbedding(
            self.head_dim
        )

        self.norm1 = nn.LayerNorm(dim)

        self.qkv = nn.Linear(
            dim,
            dim * 3
        )

        self.out = nn.Linear(
            dim,
            dim
        )

        self.norm2 = nn.LayerNorm(dim)

        self.ff = nn.Sequential(

            nn.Linear(
                dim,
                ff_dim
            ),

            nn.GELU(),

            nn.Linear(
                ff_dim,
                dim
            )

        )

    # =====================================================
    # SPLIT HEADS
    # =====================================================

    def split_heads(
        self,
        x
    ):

        B, T, D = x.shape

        x = x.view(
            B,
            T,
            self.heads,
            self.head_dim
        )

        return x.transpose(
            1,
            2
        )

    # =====================================================
    # MERGE HEADS
    # =====================================================

    def merge_heads(
        self,
        x
    ):

        B, H, T, D = x.shape

        x = x.transpose(
            1,
            2
        ).contiguous()

        return x.view(
            B,
            T,
            H * D
        )

    # =====================================================
    # FORWARD
    # =====================================================

    def forward(
        self,
        x,
        kv_cache=None
    ):

        residual = x

        x = self.norm1(x)

        qkv = self.qkv(x)

        q, k, v = qkv.chunk(
            3,
            dim=-1
        )

        q = self.split_heads(q)
        k = self.split_heads(k)
        v = self.split_heads(v)

        # ==========================================
        # ROPE POSITION OFFSET
        # ==========================================

        if kv_cache is None:

            cache_len = 0

        else:

            cache_len = kv_cache[0].size(2)

        total_len = (
            cache_len
            +
            q.size(2)
        )

        freqs = self.rope(
            total_len,
            q.device
        )

        freqs_q = freqs[
            cache_len:
        ]

        q, k = apply_rope(
            q,
            k,
            freqs_q
        )

        # ==========================================
        # KV CACHE
        # ==========================================

        if kv_cache is not None:

            old_k, old_v = kv_cache

            k = torch.cat(
                [
                    old_k,
                    k
                ],
                dim=2
            )

            v = torch.cat(
                [
                    old_v,
                    v
                ],
                dim=2
            )

            if k.size(2) > MAX_CACHE:

                k = k[:, :, -MAX_CACHE:]
                v = v[:, :, -MAX_CACHE:]

        new_cache = (
            k.detach(),
            v.detach()
        )

        # ==========================================
        # FLASH ATTENTION
        # ==========================================

        attn = scaled_dot_product_attention(

            q,

            k,

            v,

            is_causal=True

        )

        attn = self.merge_heads(
            attn
        )

        x = residual + self.out(
            attn
        )

        x = x + self.ff(
            self.norm2(x)
        )

        return (
            x,
            new_cache
        )

# =========================================================
# MODEL
# =========================================================

class HybridDNACompressor(nn.Module):

    def __init__(self):

        super().__init__()

        d_model = 256
        latent_dim = 128

        # =====================================================
        # TOKEN EMBEDDING
        # =====================================================

        self.embed = nn.Embedding(
            VOCAB_SIZE,
            d_model
        )

        # =====================================================
        # SPECIES EMBEDDING
        # =====================================================

        self.species_embed = nn.Embedding(
            16,
            d_model
        )

        # =====================================================
        # ENCODER
        # =====================================================

        self.encoder_blocks = nn.ModuleList([

            FlashAttentionBlock(
                d_model,
                heads=8,
                ff_dim=1024
            )

            for _ in range(8)

        ])

        # =====================================================
        # BOTTLENECK
        # =====================================================

        self.to_latent = nn.Linear(
            d_model,
            latent_dim
        )

        self.from_latent = nn.Linear(
            latent_dim,
            d_model
        )

        # =====================================================
        # DECODER
        # =====================================================

        self.decoder_blocks = nn.ModuleList([

            FlashAttentionBlock(
                d_model,
                heads=8,
                ff_dim=1024
            )

            for _ in range(4)

        ])

        # =====================================================
        # OUTPUT HEAD
        # =====================================================

        self.fc = nn.Sequential(

            nn.Linear(
                d_model,
                d_model
            ),

            nn.GELU(),

            nn.Linear(
                d_model,
                VOCAB_SIZE
            )

        )

        # =====================================================
        # POINTER COPY
        # =====================================================

        self.copy_query = nn.Linear(
            d_model,
            d_model
        )

        self.copy_key = nn.Linear(
            d_model,
            d_model
        )

        self.copy_gate = nn.Linear(
            d_model,
            1
        )

        # =====================================================
        # MEMORY BRANCH
        # =====================================================

        self.memory_query = nn.Linear(
            d_model,
            d_model
        )

        self.memory_key = nn.Linear(
            d_model,
            d_model
        )

        self.memory_value = nn.Linear(
            d_model,
            VOCAB_SIZE
        )

        self.memory_gate = nn.Linear(
            d_model,
            1
        )

    def forward(
        self,
        x,
        species_id=None,
        kv_cache=None
    ):

        input_tokens = x.clone()

        # =====================================================
        # EMBEDDING
        # =====================================================

        x = self.embed(x)

        if species_id is not None:

            if species_id.dim() == 0:

                species_id = species_id.unsqueeze(0)

            species_emb = self.species_embed(
                species_id
            )

            x = x + species_emb.unsqueeze(1)

        # =====================================================
        # CACHE INIT
        # =====================================================

        if kv_cache is None:

            kv_cache = [

                None

                for _ in range(
                    len(self.encoder_blocks)
                )

            ]

        new_cache = []

        # =====================================================
        # ENCODER
        # =====================================================

        for block, cache in zip(
            self.encoder_blocks,
            kv_cache
        ):

            x, updated_cache = block(
                x,
                cache
            )

            new_cache.append(
                updated_cache
            )

        # =====================================================
        # LATENT BOTTLENECK
        # =====================================================

        residual = x

        z = self.to_latent(x)

        x = self.from_latent(z)

        x = x + residual

        # =====================================================
        # DECODER
        # =====================================================

        for block in self.decoder_blocks:

            x, _ = block(x)

        # =====================================================
        # BASE DISTRIBUTION
        # =====================================================

        logits = self.fc(x)

        base_probs = F.softmax(
            logits,
            dim=-1
        )

        # =====================================================
        # POINTER COPY
        # =====================================================

        q_copy = self.copy_query(
            x
        ).float()

        k_copy = self.copy_key(
            x
        ).float()

        copy_scores = torch.matmul(

            q_copy,

            k_copy.transpose(-1, -2)

        )

        causal_mask = torch.triu(

            torch.ones(
                copy_scores.shape,
                device=copy_scores.device,
                dtype=torch.bool
            ),

            diagonal=1

        )

        copy_scores = copy_scores.masked_fill(
            causal_mask,
            -1e4
        )

        copy_attn = F.softmax(
            copy_scores,
            dim=-1
        )

        copy_probs = torch.zeros_like(
            base_probs
        )

        for symbol in range(
            VOCAB_SIZE
        ):

            symbol_mask = (

                input_tokens.unsqueeze(1)

                == symbol

            ).float()

            copy_probs[:, :, symbol] = (

                copy_attn

                * symbol_mask

            ).sum(-1)

        # =====================================================
        # MEMORY DISTRIBUTION
        # =====================================================

        memory_q = self.memory_query(
            x
        ).float()

        memory_k = self.memory_key(
            x
        ).float()

        memory_scores = torch.matmul(

            memory_q,

            memory_k.transpose(-1, -2)

        )

        memory_scores = memory_scores.masked_fill(
            causal_mask,
            -1e4
        )

        memory_attn = F.softmax(
            memory_scores,
            dim=-1
        )

        memory_logits = self.memory_value(
            x
        )

        memory_probs = F.softmax(
            memory_logits,
            dim=-1
        )

        memory_probs = torch.matmul(

            memory_attn,

            memory_probs

        )

        # =====================================================
        # MIXTURE
        # =====================================================

        copy_gate = torch.sigmoid(
            self.copy_gate(x)
        )

        memory_gate = torch.sigmoid(
            self.memory_gate(x)
        )

        final_probs = (

            copy_gate

            * base_probs

            +

            (1.0 - copy_gate)

            * (

                memory_gate

                * copy_probs

                +

                (1.0 - memory_gate)

                * memory_probs

            )

        )

        final_probs = torch.clamp(
            final_probs,
            min=1e-8
        )

        log_probs = torch.log(
            final_probs
        )

        return (

            log_probs,

            z,

            new_cache

        )

# =========================================================
# BPB
# =========================================================

def compute_bpb(log_probs, targets):

    probs = torch.exp(
        log_probs
    )

    probs = probs.gather(
        -1,
        targets.unsqueeze(-1)
    ).squeeze(-1)

    return (
        -torch.log2(
            probs + 1e-9
        )
    ).mean().item()

# =========================================================
# EVALUATION
# =========================================================

# =========================================================
# EVALUATION
# =========================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()

    total_bpb = 0.0

    num_batches = 0

    for x, y, species_id in loader:

        x = x.to(
            DEVICE,
            non_blocking=True
        )

        y = y.to(
            DEVICE,
            non_blocking=True
        )

        species_id = species_id.to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type=DEVICE.type
        ):

            log_probs, _, _ = model(
                x,
                species_id=species_id
            )

        batch_bpb = compute_bpb(
            log_probs,
            y
        )

        total_bpb += batch_bpb

        num_batches += 1

    return total_bpb / max(
        num_batches,
        1
    )

# =========================================================
# TRAINING
# =========================================================

# =========================================================
# TRAINING
# =========================================================

def train_model(
    train_loader,
    val_loader
):

    model = HybridDNACompressor().to(DEVICE)

    try:

        model = torch.compile(model)

        print("torch.compile enabled")

    except Exception:

        print("torch.compile unavailable")

    optimizer = optim.AdamW(

        model.parameters(),

        lr=LR,

        weight_decay=1e-4

    )

    scaler = torch.amp.GradScaler()

    best_val = float("inf")

    patience_ctr = 0

    for epoch in range(EPOCHS):

        model.train()

        t0 = time.time()

        total_loss = 0.0

        total_pred_loss = 0.0

        total_latent_loss = 0.0

        for x, y, species_id in train_loader:

            x = x.to(
                DEVICE,
                non_blocking=True
            )

            y = y.to(
                DEVICE,
                non_blocking=True
            )

            species_id = species_id.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.amp.autocast(
                device_type=DEVICE.type
            ):

                log_probs, z, _ = model(
                    x,
                    species_id=species_id
                )

                pred_loss = F.nll_loss(

                    log_probs.reshape(
                        -1,
                        VOCAB_SIZE
                    ),

                    y.reshape(-1)

                )

                latent_loss = (

                    torch.mean(
                        z ** 2
                    )

                    +

                    0.25 * torch.mean(
                        torch.abs(z)
                    )

                )

                loss = (

                    pred_loss

                    +

                    1e-4 * latent_loss

                )

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                1.0

            )

            scaler.step(
                optimizer
            )

            scaler.update()

            total_loss += loss.item()

            total_pred_loss += pred_loss.item()

            total_latent_loss += latent_loss.item()

        val_bpb = evaluate(
            model,
            val_loader
        )

        avg_loss = (

            total_loss

            /

            len(train_loader)

        )

        avg_pred = (

            total_pred_loss

            /

            len(train_loader)

        )

        avg_latent = (

            total_latent_loss

            /

            len(train_loader)

        )

        elapsed = (

            time.time()

            -

            t0

        )

        print(
            f"Epoch {epoch+1}/{EPOCHS}"
        )

        print(
            f"Train Loss: {avg_loss:.4f}"
        )

        print(
            f"Train NLL: {avg_pred:.4f}"
        )

        print(
            f"Train Latent: {avg_latent:.4f}"
        )

        print(
            f"Val BPB: {val_bpb:.4f}"
        )

        print(
            f"Time: {elapsed:.1f}s"
        )

        if val_bpb < best_val:

            best_val = val_bpb

            patience_ctr = 0

            torch.save(

                model.state_dict(),

                MODEL_PATH

            )

            print(
                f"New Best BPB: {best_val:.4f}"
            )

        else:

            patience_ctr += 1

            print(
                f"Patience: {patience_ctr}/{PATIENCE}"
            )

        print()

        if patience_ctr >= PATIENCE:

            print(
                "Early stopping"
            )

            break

    model.load_state_dict(

        torch.load(
            MODEL_PATH,
            map_location=DEVICE
        )

    )

    return model
# =========================================================
# COMPRESS
# =========================================================

@torch.inference_mode()
def compress(
    model,
    sequence,
    species_id=0
):

    model.eval()

    seq = torch.tensor(
        [DNA_MAP[c] for c in sequence],
        dtype=torch.long
    )

    species_id = torch.tensor(
        [species_id],
        dtype=torch.long,
        device=DEVICE
    )

    cdfs = []

    context = seq[
        :SEQ_LEN
    ].unsqueeze(0).to(DEVICE)

    with torch.amp.autocast(
        device_type=DEVICE.type
    ):

        _, _, kv_cache = model(
            context,
            species_id=species_id,
            kv_cache=None
        )

    for i in range(
        SEQ_LEN,
        len(seq),
        CHUNK_SIZE
    ):

        end_idx = min(
            i + CHUNK_SIZE,
            len(seq)
        )

        token_chunk = seq[
            i - 1:end_idx - 1
        ].unsqueeze(0).to(DEVICE)

        with torch.amp.autocast(
            device_type=DEVICE.type
        ):

            log_probs, _, kv_cache = model(
                token_chunk,
                species_id=species_id,
                kv_cache=kv_cache
            )

        probs = torch.exp(
            log_probs[0]
        ).float()

        probs = probs / (
            probs.sum(
                dim=-1,
                keepdim=True
            ) + 1e-12
        )

        cdf = torch.cumsum(
            probs,
            dim=-1
        )

        cdf = torch.cat(

            [

                torch.zeros(
                    cdf.size(0),
                    1,
                    device=cdf.device
                ),

                cdf

            ],

            dim=-1

        )

        cdf = torch.clamp(
            cdf,
            0.0,
            1.0
        )

        cdf[:, -1] = 1.0

        cdfs.append(
            cdf.cpu().contiguous()
        )

    cdfs = torch.cat(
        cdfs,
        dim=0
    )

    symbols = seq[
        SEQ_LEN:
    ].cpu().to(torch.int16)

    encoded = torchac.encode_float_cdf(
        cdfs.float(),
        symbols,
        check_input_bounds=True
    )

    return encoded, cdfs
# =========================================================
# DECOMPRESS
# =========================================================

@torch.no_grad()
def decompress(
    encoded,
    cdfs,
    seed_context
):

    cdfs = cdfs.float().cpu()

    symbols = torchac.decode_float_cdf(
        cdfs,
        encoded
    )

    decoded = [

        DNA_MAP[c]

        for c in seed_context

    ]

    decoded.extend(

        int(x)

        for x in symbols

    )

    sequence = ''.join(

        REV_DNA_MAP[token]

        for token in decoded

    )

    return sequence

# =========================================================
# RECONSTRUCTION ACCURACY
# =========================================================

def reconstruction_accuracy(
    original,
    reconstructed
):

    correct = sum(

        o == r

        for o, r in zip(
            original,
            reconstructed
        )

    )

    return (

        correct / len(original)

    ) * 100

# =========================================================
# SAVE FILES
# =========================================================

def save_compressed(
    encoded,
    filename="compressed.dnaac"
):

    with open(filename, "wb") as f:
        f.write(encoded)

def save_decompressed(
    sequence,
    filename="decompressed.fasta"
):

    with open(filename, "w") as f:

        f.write(">decoded_sequence\n")

        for i in range(
            0,
            len(sequence),
            80
        ):

            f.write(
                sequence[i:i+80] + "\n"
            )

# =========================================================
# MAIN
# =========================================================

# =========================================================
# MAIN
# =========================================================

def main():

    print("\nLoading species genomes...")
    ## For Example


    esco_seq = load_fasta("/content/EsCo")
    agph_seq = load_fasta("/content/AgPh")
    bueb_seq = load_fasta("/content/BuEb")
    yemi_seq = load_fasta("/content/YeMi")

    test_file = "/content/WaMe"

    # =====================================================
    # SPLIT EACH SPECIES
    # =====================================================

    esco_split = int(
        0.9 * len(esco_seq)
    )

    agph_split = int(
        0.9 * len(agph_seq)
    )

    bueb_split = int(
        0.9 * len(bueb_seq)
    )

    yemi_split = int(
        0.9 * len(yemi_seq)
    )

    train_esco = esco_seq[:esco_split]
    val_esco = esco_seq[esco_split:]

    train_agph = agph_seq[:agph_split]
    val_agph = agph_seq[agph_split:]

    train_bueb = bueb_seq[:bueb_split]
    val_bueb = bueb_seq[bueb_split:]

    train_yemi = yemi_seq[:yemi_split]
    val_yemi = yemi_seq[yemi_split:]

    # =====================================================
    # TRAIN DATASET
    # =====================================================

    train_dataset = torch.utils.data.ConcatDataset([

        SequenceDataset(
            train_esco,
            SEQ_LEN,
            STRIDE,
            species_id=0
        ),

        SequenceDataset(
            train_agph,
            SEQ_LEN,
            STRIDE,
            species_id=1
        ),

        SequenceDataset(
            train_bueb,
            SEQ_LEN,
            STRIDE,
            species_id=2
        ),

        SequenceDataset(
            train_yemi,
            SEQ_LEN,
            STRIDE,
            species_id=3
        )

    ])

    # =====================================================
    # VALIDATION DATASET
    # =====================================================

    val_dataset = torch.utils.data.ConcatDataset([

        SequenceDataset(
            val_esco,
            SEQ_LEN,
            STRIDE,
            species_id=0
        ),

        SequenceDataset(
            val_agph,
            SEQ_LEN,
            STRIDE,
            species_id=1
        ),

        SequenceDataset(
            val_bueb,
            SEQ_LEN,
            STRIDE,
            species_id=2
        ),

        SequenceDataset(
            val_yemi,
            SEQ_LEN,
            STRIDE,
            species_id=3
        )

    ])

    print(
        f"Train Samples: {len(train_dataset)}"
    )

    print(
        f"Validation Samples: {len(val_dataset)}"
    )

    # =====================================================
    # DATALOADERS
    # =====================================================

    train_loader = DataLoader(

        train_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        pin_memory=True,

        num_workers=4,

        persistent_workers=True

    )

    val_loader = DataLoader(

        val_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        pin_memory=True,

        num_workers=4,

        persistent_workers=True

    )

    # =====================================================
    # TRAIN
    # =====================================================

    print("\nTraining...")

    model = train_model(
        train_loader,
        val_loader
    )

    # =====================================================
    # TEST GENOME
    # =====================================================

    print("\nLoading test genome...")

    test_seq = load_fasta(
        test_file
    )

    # =====================================================
    # COMPRESS
    # =====================================================

    print("\nCompressing...")

    start_time = time.time()

    encoded, cdfs = compress(
        model,
        test_seq,
        species_id=0
    )

    compression_time = (
        time.time()
        -
        start_time
    )

    save_compressed(
        encoded
    )

    neural_bpb = (
        len(encoded) * 8
    ) / len(test_seq)

    # =====================================================
    # DECOMPRESS
    # =====================================================

    print("\nDecompressing...")

    start_time = time.time()

    decoded_seq = decompress(
        encoded,
        cdfs,
        test_seq[:SEQ_LEN]
    )

    decompression_time = (
        time.time()
        -
        start_time
    )

    save_decompressed(
        decoded_seq
    )

    # =====================================================
    # METRICS
    # =====================================================

    recon_acc = reconstruction_accuracy(
        test_seq,
        decoded_seq
    )

    lossless_ok = (
        decoded_seq == test_seq
    )

    gzip_bpb = (
        len(
            gzip.compress(
                test_seq.encode()
            )
        ) * 8
    ) / len(test_seq)

    bzip_bpb = (
        len(
            bz2.compress(
                test_seq.encode()
            )
        ) * 8
    ) / len(test_seq)

    print("\n========== FINAL RESULTS ==========")

    print(
        f"Neural BPB: {neural_bpb:.4f}"
    )

    print(
        f"gzip BPB: {gzip_bpb:.4f}"
    )

    print(
        f"bzip2 BPB: {bzip_bpb:.4f}"
    )

    print(
        f"Compression Ratio: {8/neural_bpb:.4f}x"
    )

    print(
        f"Compression Time: {compression_time:.2f} sec"
    )

    print(
        f"Decompression Time: {decompression_time:.2f} sec"
    )

    print(
        f"Reconstruction Accuracy: {recon_acc:.4f}%"
    )

    print(
        f"Lossless Decode: {lossless_ok}"
    )


if __name__ == "__main__":
    main()

Using device: cuda
NVIDIA A100-SXM4-80GB

Loading species genomes...
Train Samples: 16765
Validation Samples: 1834

Training...
torch.compile enabled


/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is dis

Epoch 1/40
Train Loss: 1.3709
Train NLL: 1.3708
Train Latent: 0.8610
Val BPB: 1.9658
Time: 179.9s
New Best BPB: 1.9658

Epoch 2/40
Train Loss: 1.3556
Train NLL: 1.3556
Train Latent: 0.4404
Val BPB: 1.9475
Time: 51.8s
New Best BPB: 1.9475

Epoch 3/40
Train Loss: 1.3446
Train NLL: 1.3445
Train Latent: 0.5314
Val BPB: 1.9352
Time: 51.9s
New Best BPB: 1.9352

Epoch 4/40
Train Loss: 1.3384
Train NLL: 1.3384
Train Latent: 0.5294
Val BPB: 1.9263
Time: 51.9s
New Best BPB: 1.9263

Epoch 5/40
Train Loss: 1.3340
Train NLL: 1.3340
Train Latent: 0.4960
Val BPB: 1.9213
Time: 52.1s
New Best BPB: 1.9213

Epoch 6/40
Train Loss: 1.3302
Train NLL: 1.3302
Train Latent: 0.4688
Val BPB: 1.9132
Time: 52.1s
New Best BPB: 1.9132

Epoch 7/40
Train Loss: 1.3245
Train NLL: 1.3245
Train Latent: 0.4425
Val BPB: 1.9027
Time: 52.1s
New Best BPB: 1.9027

Epoch 8/40
Train Loss: 1.3162
Train NLL: 1.3162
Train Latent: 0.4066
Val BPB: 1.8906
Time: 52.8s
New Best BPB: 1.8906

Epoch 9/40
Train Loss: 1.3103
Train NLL: 1.3102

/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7836: UserWarning: 
Online softmax is dis


Decompressing...

========== FINAL RESULTS ==========
Neural BPB: 2.0449
gzip BPB: 2.2517
bzip2 BPB: 2.1678
Compression Ratio: 3.9122x
Compression Time: 660.24 sec
Decompression Time: 18.97 sec
Reconstruction Accuracy: 100.0000%
Lossless Decode: True
